# 🤖 Simple RAG Demo — HR Policy Assistant

**RAG** stands for **Retrieval-Augmented Generation**. In plain English:

1. We take a document (here, an HR policy handbook).
2. We chop it into small pieces and turn each piece into a list of numbers (an "embedding") that captures its meaning.
3. We store those pieces in a searchable database (a "vector store").
4. When a user asks a question, we **search** the database for the most relevant pieces, and hand them to an AI model to **generate** a final answer.

That's it. No magic — just search + a language model.

**What we use in this notebook:**
- 📄 Data: `data/hr_policy.txt` (a sample HR policy document)
- 🔢 Embeddings: **Jina AI**
- 🗄️ Vector store: **FAISS**
- 🧠 LLM: **Groq** (fast + free-tier friendly)
- 🕸️ Framework: **LangChain** (`create_agent`)

> Before running: make sure the notebook's kernel is set to the `ragenv` virtual environment (top-right corner of VS Code / Jupyter), and that a `.env` file with `GROQ_API_KEY` and `JINA_API_KEY` exists in this folder.

In [1]:
import os 
from dotenv import load_dotenv

In [ ]:
# LLM MODEL - text as input / text as clear_output

# EMB MDOEL - text as ipnput / embedding as outptu

In [2]:
# langchain

from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter 

from langchain_community.embeddings import JinaEmbeddings


In [4]:
load_dotenv()

True

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """Rule 1: Always greet the customer with a polite hello.
Rule 2: Never give out internal server passwords.
Rule 3: Escalate to a human if the user gets angry."""

# Setup the smart splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=60,       # Maximum characters allowed per chunk
    chunk_overlap=10     # How many characters to repeat from the previous chunk
)

# Chop the text
chunks = splitter.split_text(text)

# Print the results
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: '{chunk}'")


Chunk 1: 'Rule 1: Always greet the customer with a polite hello.'
Chunk 2: 'Rule 2: Never give out internal server passwords.'
Chunk 3: 'Rule 3: Escalate to a human if the user gets angry.'


In [ ]:
groq_key = os.getenv("GROQ_API_KEY")  # LLM 
jina_key = os.getenv("JINA_API_KEY")   # EMBEDDING MODEL

print("ENV VAR LOADED")

ENV VAR LOADED


LOAD THE TEXT

In [58]:
DATA_FILE_PATH = os.path.join("data" , "hr_policy.txt")


In [59]:
loader = TextLoader(DATA_FILE_PATH , encoding="utf-8")
documents = loader.load()

In [60]:
len(documents)

1

In [61]:
documents[0].metadata

{'source': 'data\\hr_policy.txt'}

LANGCHAIN DOCUMENT

PAGE CONTENT   - actual data 

METADATA  - you have info of your data 

In [62]:
print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: data\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2603

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


In [63]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [64]:
len(chunks)

9

NOW EACH SPILTED CHUNK IS A DOCUMENT - page content and metadata

In [65]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [66]:
print(chunks[2])

page_content='2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.' metadata={'source': 'data\\hr_policy.txt'}


In [67]:
print(chunks[6])

page_content='6. CODE OF CONDUCT
Employees are expected to maintain professionalism and respect in the workplace.
Harassment, discrimination, or any form of workplace misconduct will not be tolerated
and may result in disciplinary action, including termination.
All employees must complete an annual Code of Conduct training.' metadata={'source': 'data\\hr_policy.txt'}


EMBEDDING

In [68]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

In [69]:
embeddings_model.model_name

'jina-embeddings-v2-base-en'

In [70]:
embeddings_model

JinaEmbeddings(session=<requests.sessions.Session object at 0x00000195D1B1EAE0>, model_name='jina-embeddings-v2-base-en', jina_api_key=None)

In [71]:
query_vector = embeddings_model.embed_query("What is LangChain?")

query_vector

[-0.00962079,
 -0.03576203,
 -0.00614623,
 -0.00571117,
 -0.02768769,
 0.01993371,
 -0.00105305,
 -0.02854793,
 0.06249061,
 0.07040081,
 -0.02841345,
 -0.04780927,
 -0.01529932,
 -0.00992187,
 -0.05072023,
 0.09596659,
 -0.01636127,
 -0.01124633,
 0.05107618,
 -0.03327032,
 0.00255549,
 -0.02981356,
 -0.04322135,
 -0.02895135,
 0.07470396,
 0.02174911,
 0.07038499,
 0.05853551,
 0.02928357,
 0.0218638,
 -0.01902799,
 -0.02199037,
 -0.03330196,
 0.00566049,
 0.00278785,
 -0.03365001,
 -0.01911754,
 -0.04297021,
 0.02452163,
 0.07486217,
 -0.04831553,
 -0.01027338,
 0.00664309,
 0.08210792,
 -0.04645663,
 -0.00460572,
 -0.01889748,
 0.02273393,
 -0.05072616,
 0.02377412,
 0.00210819,
 -0.04055957,
 0.03049383,
 -0.05992771,
 0.02606017,
 0.01649475,
 0.05625737,
 -0.04516726,
 -0.02222965,
 -0.0238374,
 -0.01881837,
 0.03407716,
 -0.02861121,
 0.01495622,
 0.01808569,
 0.02791907,
 -0.01657979,
 -0.08058916,
 -0.01703858,
 0.0062629,
 -0.01701633,
 0.0159707,
 0.02659411,
 0.05451712,
 

text -- emeddings - vector db


In [72]:
from langchain_community.vectorstores import FAISS 

In [73]:
vector_store = FAISS.from_documents(chunks , embeddings_model)

In [74]:
print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [75]:
test_query = "How many sick leaves employees get"

VECTOR SEARCH 

In [79]:
top_matches = vector_store.similarity_search(test_query , k=3)

In [80]:
top_matches

[Document(id='5bc1a118-f615-4020-9537-de843798e324', metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'),
 Document(id='4d4e0e4e-6a7f-4721-a14a-1c3b0498fbe6', metadata={'source': 'data\\hr_policy.txt'}, page_content='7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.'),
 Document(id='5a13a1cd-96ec-4dd1-9f51-05b4f76a0fa6', metadata={'source': 'data\\hr_policy.txt

In [81]:
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

--- Match 3 ---
4. NOTICE PERIOD
Employees who wish to resign must serve a notice period of 30 days.
During the probation period, the notice period is reduced to 15 days.
The company may waive the notice period at its discretion, with full and final se

END OF DATA INGESTING PIPELINE